# EdgeDeID — Baseline Comparison & Efficiency Benchmark

Evaluates our **Qwen3-0.6B** de-identification student (SFT and SFT+GRPO) against popular
PHI/PII baselines, on the **`en_data.csv`** benchmark — 100 real radiology reports manually
de-identified by the supervisor, with the shared rule-based `preprocess_text` applied to build
the gold XML. This is the headline benchmark for the paper.

**Two inference backends, matching how each model is meant to run:**
- **Our Qwen models** are served with **vLLM** (identical to `evals/ext_eval/en_eval.ipynb`) and
  queried over the OpenAI-compatible API. vLLM batches correctly server-side.
- **Baselines** (rule-based / NER) load **in-process** via `transformers`/`spaCy`.

The notebook is **cell-by-cell** — every model runs in a visible loop so you can watch metrics
change. Sections:

| § | Purpose |
|---|---|
| 0 | Setup & dependencies |
| 1 | Load `en_data.csv` |
| 2 | Baseline registry + helpers |
| 3 | Smoke test (load + parse) |
| 4 | Run baselines (accuracy + latency) |
| 5 | Run our Qwen models via vLLM (accuracy) |
| 6 | Combined comparison table + per-entity |
| 7 | Accuracy plots |
| 8 | (Optional) GPU-vs-CPU latency micro-benchmark for our models |
| 9 | Paper-ready master table (CSV + LaTeX) |

| Model | Type | Source |
|---|---|---|
| Presidio | Rule-based | `presidio-analyzer` |
| GLiNER | Zero-shot NER | `urchade/gliner_mediumv2.1` |
| RoBERTa-i2b2 | Fine-tuned NER | `obi/deid_roberta_i2b2` |
| BERT-i2b2 | Fine-tuned NER | `obi/deid_bert_i2b2` |
| scispaCy | Medical NER | `en_core_web_lg` *(optional — numpy-pinned)* |
| OpenMed-PII | Fine-tuned NER | `OpenMed/OpenMed-PII-SuperClinical-Large-434M-v1` |
| OPF | Span-decode NER | `openai/privacy-filter` *(optional — from source)* |
| Qwen3-0.6B-Base | Generative (zero-shot) | `Qwen/Qwen3-0.6B-Base` *(vLLM)* |
| **EdgeDeID-SFT** | **Ours** | `EdgeDeID/Qwen3-0.6B-Base_SFT` *(vLLM)* |
| **EdgeDeID-GRPO** | **Ours** | `EdgeDeID/Qwen3-0.6B-GRPO` *(vLLM)* |

In [ ]:
# --- repo bootstrap: makes this notebook runnable from anywhere ---
import os, sys
ROOT = os.getcwd()
while ROOT != os.path.dirname(ROOT) and not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = os.path.dirname(ROOT)
if not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = "/kaggle/input/datasets/sinadaneshgar/edgedeid"  # Kaggle dataset fallback
os.chdir(ROOT); sys.path.insert(0, ROOT)
print("repo root:", ROOT)

## 0 — Setup & dependencies

Repo on the path, optional installs, imports, HF auth.

> Do **not** install `scispacy` here — it pins an old numpy and breaks pandas
> (`numpy.dtype size changed`). Install it only in a throwaway session.

In [ ]:
# REPO_DIR is resolved by the bootstrap cell above.
REPO_DIR = ROOT
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("REPO_DIR =", REPO_DIR)

In [ ]:
# Baseline + serving dependencies. The harness skips any model whose deps are missing.
# accelerate: transformers GPU loading | vllm + openai: serve/query our Qwen models.
!pip install -q --no-warn-conflicts accelerate gliner presidio-analyzer presidio-anonymizer openai
# vLLM is large; install on the serving machine if not present:
# !pip install -q vllm
# scispaCy (optional, separate session + kernel restart):
# !pip install -q scispacy https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_web_lg-0.5.4.tar.gz
# OPF / OpenAI privacy-filter (optional, only if INCLUDE_OPF=True):
# !git clone -q https://github.com/openai/privacy-filter /tmp/privacy-filter && pip install -q -e /tmp/privacy-filter

In [ ]:
import gc, time, json, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch

from edgedeid.eval import compute_metrics, print_report

RESULTS_DIR = Path("artifacts/baselines"); RESULTS_DIR.mkdir(exist_ok=True)
PRED_DIR = RESULTS_DIR / "predictions"; PRED_DIR.mkdir(exist_ok=True)

HAS_CUDA = torch.cuda.is_available()
print("torch", torch.__version__, "| CUDA:", HAS_CUDA,
      "|", torch.cuda.get_device_name(0) if HAS_CUDA else "CPU only")

In [ ]:
# ── Hugging Face auth (only needed for gated/private repos) ────────────────────
def setup_hf(token=None):
    token = token or os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if token is None:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if token is None:
        print("No HF token found - public repos only."); return
    os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    from huggingface_hub import login
    login(token=token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub.")

setup_hf()

## 1 — Load `en_data.csv`

The 100 manually-annotated radiology reports. `preprocess_text` applies the rule-based
normalisation and `convert_to_xml` builds the gold `<phi_entities>` XML — exactly the pipeline
the supervisor's `en_eval.ipynb` uses, so the gold labels are identical.

The gold `<date>` may show `<formatted>2024-??-??</formatted>`: intentional, not a bug — when the
source states only a partial date (e.g. year), the unknown parts are filled with `?`.

In [ ]:
from edgedeid.data.preprocessing import preprocess_text
from edgedeid.data.xml_utils import convert_to_xml

raw_df = pd.read_csv(f"{REPO_DIR}/evals/ext_eval/en_data.csv")
raw_df = preprocess_text(raw_df, input_col="Report", output_col="DeID")
raw_df["GT-XML"] = raw_df["DeID"].apply(convert_to_xml)

NOTE_COL = "Report"
for tag in ["<anonymized_report>", "</anonymized_report>", "<anonymized_report/>",
            "<raw_report>", "</raw_report>", "<raw_report/>"]:
    raw_df[NOTE_COL] = raw_df[NOTE_COL].str.replace(tag, "", regex=False).str.strip()

notes   = raw_df[NOTE_COL].tolist()
gt_xmls = raw_df["GT-XML"].tolist()
print(f"{len(notes)} evaluation samples (en_data.csv)")

In [ ]:
# Sanity-check one sample.
print("NOTE:\n", notes[0][:400], "\n")
print("GOLD XML:\n", gt_xmls[0][:400])

## 2 — Baseline registry + helpers

A lazy `(label, factory)` registry of the **in-process** baselines (generative models are run
separately via vLLM in §5). Nothing loads until a loop calls the factory, so one model occupies
the GPU at a time. Plus small single-purpose helpers.

In [ ]:
from edgedeid.baselines import (
    PresidioBaseline, GLiNERBaseline, RobertaDeidBaseline, ScispaCyBaseline,
    BertDeidBaseline, OpenMedPIIBaseline, OPFBaseline,
)

INCLUDE_SCISPACY = True    # needs en_core_web_lg (numpy-pinned; install separately)
INCLUDE_OPF      = True    # needs openai/privacy-filter from source

BASELINES = [
    ("Presidio",     lambda: PresidioBaseline(score_threshold=0.5)),
    ("GLiNER",       lambda: GLiNERBaseline(model_name="urchade/gliner_mediumv2.1", threshold=0.5)),
    ("RoBERTa-i2b2", lambda: RobertaDeidBaseline(model_name="obi/deid_roberta_i2b2")),
    ("BERT-i2b2",    lambda: BertDeidBaseline(model_name="obi/deid_bert_i2b2")),
    ("OpenMed-PII",  lambda: OpenMedPIIBaseline(model_name="OpenMed/OpenMed-PII-SuperClinical-Large-434M-v1")),
]
if INCLUDE_SCISPACY:
    BASELINES.append(("scispaCy", lambda: ScispaCyBaseline(model_name="en_core_web_lg")))
if INCLUDE_OPF:
    BASELINES.append(("OPF", lambda: OPFBaseline()))

print(f"{len(BASELINES)} baselines registered:", [b for b, _ in BASELINES])

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def count_params(model):
    # Best-effort parameter count for any wrapper (None if unknown).
    def _n(o):
        if hasattr(o, "num_parameters"):
            try: return int(o.num_parameters())
            except Exception: pass
        if hasattr(o, "parameters"):
            try: return sum(p.numel() for p in o.parameters())
            except Exception: pass
        return None
    for attr in ("model", "_model"):
        if hasattr(model, attr) and (n := _n(getattr(model, attr))): return n
    if hasattr(model, "_pipe") and (n := _n(getattr(model._pipe, "model", None))): return n
    return None

def on_gpu(model):
    return str(getattr(model, "device", "cpu")).startswith("cuda")

def free_model(model):
    # Drop references and release GPU memory before the next model loads.
    for attr in ("model", "_model", "_pipe", "_opf", "_analyzer", "_nlp"):
        if hasattr(model, attr):
            try: setattr(model, attr, None)
            except Exception: pass
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def evaluate(preds, supported=None):
    # compute_metrics scoped to what the model can emit (None = full schema).
    return compute_metrics(preds, gt_xmls, source_notes=notes, supported_entities=supported)

def result_row(name, r, n_params, sec_per_note):
    return {
        "Model": name,
        "Micro-P": round(r.micro_precision, 4), "Micro-R": round(r.micro_recall, 4),
        "Micro-F1": round(r.micro_f1, 4), "F-b(2)": round(r.f_beta, 4),
        "Macro-F1": round(r.macro_f1, 4), "Zero-leak": round(r.zero_leak_rate, 4),
        "XML Parse": round(r.xml_parse_rate, 4),
        "Halluc.": round(r.hallucination_rate, 4),
        "Token Reduction": (round(r.token_reduction_rate, 4)
                            if r.token_reduction_rate == r.token_reduction_rate else None),
        "#Params (M)": round(n_params / 1e6, 1) if n_params else None,
        "Sec/Note": round(sec_per_note, 4) if sec_per_note is not None else None,
    }

def per_entity_frame(results):
    tags = sorted({t for r in results.values() for t in r.per_entity})
    return pd.DataFrame([
        {"Entity": e, **{name: (round(r.per_entity[e].f1, 4) if e in r.per_entity else None)
                         for name, r in results.items()}}
        for e in tags
    ]).set_index("Entity")

print("helpers ready")

## 3 — Smoke test

Builds each baseline and checks it returns parseable `<phi_entities>` XML on 2 notes — catches
missing deps / bad checkpoints in seconds instead of after a full pass.

In [ ]:
from edgedeid.eval.metrics import _extract_phi_block

SMOKE_NOTES = notes[:2]
for label, factory in BASELINES:
    try:
        m = factory()
    except Exception as e:
        print(f"  x {label}: load failed - {type(e).__name__}: {e}"); continue
    try:
        out = m.predict_xml(SMOKE_NOTES[0])
        ok = isinstance(out, str) and _extract_phi_block(out) is not None
        print(f"  {'ok' if ok else 'x '} {label}: {'parseable XML' if ok else 'NO valid <phi_entities>'}")
    finally:
        free_model(m)

## 4 — Run baselines  (accuracy + latency)

Explicit loop: load -> sequential `predict_xml` -> metrics -> free. Sequential timing is the
realistic per-note latency for these models. Writes per-model predictions to `results/predictions/`.

In [ ]:
baseline_results, baseline_rows = {}, {}
for label, factory in BASELINES:
    print(f"\n{'='*70}\n> {label}\n{'='*70}")
    try:
        model = factory()
    except Exception as e:
        import traceback; print(f"Skipping {label}: {type(e).__name__}: {e}"); traceback.print_exc(); continue

    preds, t0 = [], time.perf_counter()
    for note in tqdm(notes, desc=label):
        preds.append(model.predict_xml(note))
    spn = (time.perf_counter() - t0) / max(len(notes), 1)

    r = evaluate(preds, supported=model.supported_entities)
    baseline_results[label] = r
    baseline_rows[label] = result_row(label, r, count_params(model), spn)
    print_report(r, title=label)
    print(f"   {spn*1000:.1f} ms/note  ·  {baseline_rows[label]['#Params (M)']} M params")
    pd.Series(preds, name="prediction").to_csv(PRED_DIR / f"pred_{label}.csv", index=False)
    free_model(model)

print("\nBaselines done:", list(baseline_results))

## 5 — Our Qwen models via vLLM  (accuracy)

Serve **one model at a time** with vLLM, then run the cell below. This matches the supervisor's
`en_eval.ipynb` exactly (same prompt builder, same OpenAI-API querying via `pipeline.eval.batch_inference`).

Launch the server in a terminal (point `--model` at the local checkpoint or HF id):

```bash
# EdgeDeID-SFT
VLLM_USE_V1=0 CUDA_VISIBLE_DEVICES=0 python -m vllm.entrypoints.openai.api_server \
  --model EdgeDeID/Qwen3-0.6B-Base_SFT --port 9995 \
  --tensor-parallel-size 1 --max-model-len 4096 --generation-config vllm --dtype bfloat16
```

Then set `SERVED` below to match `--model` / display name and run. Repeat for each of:
`Qwen/Qwen3-0.6B-Base` (zero-shot), `EdgeDeID/Qwen3-0.6B-Base_SFT`, `EdgeDeID/Qwen3-0.6B-GRPO`.

In [ ]:
from openai import OpenAI
from edgedeid.eval import batch_inference

VLLM_PORT = 9995
client = OpenAI(api_key="EMPTY", base_url=f"http://localhost:{VLLM_PORT}/v1")

# ── set these to the currently-served model, then run ─────────────────────────
SERVED_ID   = "EdgeDeID/Qwen3-0.6B-Base_SFT"   # must equal the vLLM --model argument
SERVED_NAME = "EdgeDeID-SFT"                    # display name in the tables
SERVED_PARAMS_M = 596.0                         # Qwen3-0.6B param count (M), for the size column

our_results = globals().get("our_results", {})
our_rows    = globals().get("our_rows", {})

t0 = time.perf_counter()
preds = batch_inference(notes, client, SERVED_ID, temperature=0.0, max_tokens=2048, max_workers=64)
spn = (time.perf_counter() - t0) / max(len(notes), 1)   # wall-clock per note (server-batched)

r = evaluate(preds, supported=None)                     # our models can emit the full schema
our_results[SERVED_NAME] = r
our_rows[SERVED_NAME] = result_row(SERVED_NAME, r, int(SERVED_PARAMS_M * 1e6), spn)
print_report(r, title=SERVED_NAME)
pd.Series(preds, name="prediction").to_csv(PRED_DIR / f"pred_{SERVED_NAME}.csv", index=False)
print(f"   {spn*1000:.1f} ms/note (vLLM, server-batched)")
print("Collected so far:", list(our_results))

## 6 — Combined comparison table

Merges baselines (§4) and our vLLM models (§5) into one table + per-entity F1, saved to CSV.

In [ ]:
all_results = {**baseline_results, **globals().get("our_results", {})}
all_rows    = {**baseline_rows,    **globals().get("our_rows", {})}

comparison_df = pd.DataFrame(list(all_rows.values())).set_index("Model")
comparison_df.to_csv(RESULTS_DIR / "comparison.csv")

entity_df = per_entity_frame(all_results)
entity_df.to_csv(RESULTS_DIR / "per_entity.csv")
with open(RESULTS_DIR / "results.pkl", "wb") as f:
    pickle.dump(all_results, f)

print("Saved comparison.csv, per_entity.csv, results.pkl")
comparison_df

## 7 — Accuracy plots

Saved as PNGs in `results/`.
1. Per-entity **F1** heatmap (grey = entity the model can't emit).
2. Per-entity **recall** heatmap (the safety / leak view).
3. **Precision–Recall** scatter with F1 iso-curves.
4. **Headline** metric bars.
5. **Params vs F1** efficiency frontier (* = ours).

In [ ]:
RESULTS = all_results
model_names = list(RESULTS)
ALL_TAGS = ["name", "age", "date", "place", "address", "phone", "email", "id", "url", "device"]

def _entity_matrix(attr):
    M = np.full((len(model_names), len(ALL_TAGS)), np.nan)
    for i, mn in enumerate(model_names):
        pe = RESULTS[mn].per_entity
        for j, t in enumerate(ALL_TAGS):
            if (m := pe.get(t)) is not None:
                M[i, j] = getattr(m, attr)
    return M

def _heatmap(M, title, fname, cmap_name):
    cmap = plt.get_cmap(cmap_name).copy(); cmap.set_bad("0.85")
    fig, ax = plt.subplots(figsize=(11, 1.6 + 0.5 * len(model_names)))
    im = ax.imshow(M, cmap=cmap, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(ALL_TAGS))); ax.set_xticklabels(ALL_TAGS, rotation=40, ha="right")
    ax.set_yticks(range(len(model_names))); ax.set_yticklabels(model_names)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            ax.text(j, i, "N/A" if np.isnan(v) else f"{v:.2f}", ha="center", va="center",
                    fontsize=8, color="0.4" if np.isnan(v) else ("white" if v < 0.5 else "black"))
    ax.set_title(title); fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    plt.tight_layout(); plt.savefig(RESULTS_DIR / fname, dpi=150, bbox_inches="tight"); plt.show()

_heatmap(_entity_matrix("f1"), "Per-entity F1  (grey = unsupported by model)",
         "plot_f1_heatmap.png", "viridis")

In [ ]:
_heatmap(_entity_matrix("recall"), "Per-entity recall  (red = PHI leaking, green = caught)",
         "plot_recall_heatmap.png", "RdYlGn")

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 6.8))
pp = np.linspace(0.001, 1, 400)
for f1 in (0.2, 0.4, 0.6, 0.8):
    rr = np.where(2*pp - f1 > 0, f1*pp/(2*pp - f1), np.nan); rr[(rr <= 0) | (rr > 1)] = np.nan
    ax.plot(pp, rr, "--", color="0.7", lw=0.7)
    ax.text(0.985, f1*0.985/(2*0.985 - f1), f"F1={f1}", color="0.5", fontsize=7, va="bottom", ha="right")
for mn in model_names:
    r = RESULTS[mn]
    ax.scatter(r.micro_precision, r.micro_recall, s=90, zorder=3)
    ax.annotate(mn, (r.micro_precision, r.micro_recall), textcoords="offset points", xytext=(6, 5), fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
ax.set_xlabel("Micro precision"); ax.set_ylabel("Micro recall")
ax.set_title("Precision-Recall trade-off (micro, F1 iso-curves)"); ax.grid(alpha=0.25)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "plot_pr_scatter.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
headline = [("Micro-F1", "micro_f1"), ("Micro-Recall", "micro_recall"),
            ("Zero-leak", "zero_leak_rate"), ("Macro-F1", "macro_f1")]
x = np.arange(len(headline)); w = 0.8 / max(len(model_names), 1)
fig, ax = plt.subplots(figsize=(10, 4.8))
for i, mn in enumerate(model_names):
    bars = ax.bar(x + i*w, [getattr(RESULTS[mn], a) for _, a in headline], w, label=mn)
    ax.bar_label(bars, fmt="%.2f", fontsize=6, padding=2)
ax.set_xticks(x + w*(len(model_names)-1)/2); ax.set_xticklabels([n for n, _ in headline])
ax.set_ylim(0, 1.05); ax.set_ylabel("score"); ax.set_title("Headline metrics by model")
ax.legend(fontsize=7, ncol=2); plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot_headline_bars.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for mn in model_names:
    p = comparison_df.loc[mn, "#Params (M)"] if mn in comparison_df.index else None
    f1 = RESULTS[mn].micro_f1
    if p and p == p:
        ours = mn.startswith("EdgeDeID")
        ax.scatter(p, f1, s=140 if ours else 80, marker="*" if ours else "o",
                   zorder=3, edgecolor="k" if ours else "none")
        ax.annotate(mn, (p, f1), textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("Parameters (M, log)"); ax.set_ylabel("Micro-F1")
ax.set_title("Accuracy vs model size  (* = ours)"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "plot_params_vs_f1.png", dpi=150, bbox_inches="tight"); plt.show()

## 8 — (Optional) GPU-vs-CPU latency micro-benchmark for our model

vLLM (§5) is GPU-only, so to compare **GPU vs CPU edge latency** for our 0.6B model we time the
`transformers` wrapper directly: **GPU batched** vs **CPU sequential**, on a small `LAT_N` sample.
This is a deployment-cost study, not an accuracy run (accuracy comes from §5).

Skip this section if you only need accuracy.

In [ ]:
# Requires: transformers. Times one checkpoint on GPU (batched) and CPU (sequential).
# Uses QwenDeidBaseline.predict_xml_batch (defined in baselines/qwen_deid.py).
from edgedeid.baselines import QwenDeidBaseline

LAT_CKPT = "EdgeDeID/Qwen3-0.6B-Base_SFT"
LAT_N    = 32
lat_notes = notes[:LAT_N]
lat_rows = []

if HAS_CUDA:
    m = QwenDeidBaseline(LAT_CKPT, name="lat-gpu")          # loads on cuda
    # validate batched == sequential before trusting batched timing
    ok = m.predict_xml_batch(lat_notes[:2]) == [m.predict_xml(n) for n in lat_notes[:2]]
    print("batched == sequential:", ok)
    t0 = time.perf_counter()
    if ok:
        for i in range(0, len(lat_notes), 16): m.predict_xml_batch(lat_notes[i:i+16])
    else:
        for n in lat_notes: m.predict_xml(n)
    lat_rows.append({"Device": "GPU", "Mode": "batched" if ok else "sequential",
                     "ms/note": round((time.perf_counter()-t0)/len(lat_notes)*1000, 1)})
    free_model(m)

os.environ["CUDA_VISIBLE_DEVICES"] = ""                     # force CPU for the next load
print("Note: CPU timing is slow; LAT_N kept small. (Restart kernel to use GPU again.)")

In [ ]:
# CPU sequential timing (run after the GPU cell; CUDA hidden above).
import importlib, torch as _t
importlib.reload(_t)
m = QwenDeidBaseline(LAT_CKPT, name="lat-cpu")              # loads on CPU
t0 = time.perf_counter()
for n in tqdm(lat_notes, desc="CPU"): m.predict_xml(n)
lat_rows.append({"Device": "CPU", "Mode": "sequential",
                 "ms/note": round((time.perf_counter()-t0)/len(lat_notes)*1000, 1)})
free_model(m)

lat_df = pd.DataFrame(lat_rows)
lat_df.to_csv(RESULTS_DIR / "latency_micro.csv", index=False)
print(f"\nLatency on {LAT_N} notes, model={LAT_CKPT}:")
lat_df

## 9 — Paper-ready master table

Sorts by micro-F1 and exports **CSV** + **LaTeX** (`results/master_table.tex`).

In [ ]:
master = pd.read_csv(RESULTS_DIR / "comparison.csv").set_index("Model").sort_values("Micro-F1", ascending=False)
master.to_csv(RESULTS_DIR / "master_table.csv")

paper_cols = [c for c in ["Micro-P", "Micro-R", "Micro-F1", "Macro-F1", "Zero-leak",
                          "Halluc.", "#Params (M)", "Sec/Note"] if c in master.columns]
master[paper_cols].to_latex(RESULTS_DIR / "master_table.tex", float_format="%.3f", bold_rows=True,
                            caption="PHI de-identification on en_data.csv: accuracy, size and latency.",
                            label="tab:results")
print("Saved master_table.csv and master_table.tex")

best = master.index[0]
print(f"\nBest micro-F1: {best} ({master.loc[best, 'Micro-F1']:.3f})")
for m in ["EdgeDeID-SFT", "EdgeDeID-GRPO"]:
    if m in master.index:
        row = master.loc[m]
        print(f"  {m}: F1={row['Micro-F1']:.3f}  Zero-leak={row['Zero-leak']:.3f}  Params={row.get('#Params (M)')}M")
master[paper_cols]